# EVALUATION20 — Report per il relatore (9 novembre 2026)

Questa è una copia autonoma del notebook di **sola analisi e archiviazione**. Usa il codice sperimentale già congelato, le 295 diagnosi valutate e i due archivi storici Microsoft RCA-agent.

**Non esegue alcuna chiamata API LLM, non ripete diagnosi, non modifica risultati originali.**

Produce su Google Drive sei nuovi file in `Tesi_OpenRCA/data/reports/relatore_2026-11-09/`: relazione Markdown, risultati per 15 configurazioni, ablazioni appaiate, baseline originale sui 20 casi, dettaglio dei 300 tentativi e manifest SHA256 del report.

La prima esecuzione crea i sei file con modalità **create-only**. Una seconda esecuzione viene bloccata per evitare sovrascritture.

**Prima di procedere:** assicurarsi che il repository sia al commit sperimentale congelato e che `/content/evaluation20_gemini_venv/bin/python` esista. Se Colab si è disconnesso, ripristinare l'ambiente senza eseguire di nuovo le 300 run.

In [ ]:
# Collega Drive e verifica che l'ambiente esista. Nessuna API LLM.
from pathlib import Path
from google.colab import drive

if not Path('/content/drive/MyDrive/Tesi_OpenRCA').is_dir():
    drive.mount('/content/drive')

REPO = Path('/content/Tesi_OpenRCA_Evaluation20')
PYTHON = Path('/content/evaluation20_gemini_venv/bin/python')
assert REPO.is_dir(), 'Repository Colab assente: ripristina il clone al commit congelato'
assert PYTHON.is_file(), 'Ambiente Python isolato assente: ripristina il venv già utilizzato'
print('Drive: OK')
print('Repository congelato presente: OK')
print('Ambiente Python isolato presente: OK')

## Generazione e verifica del report

La cella seguente verifica le fingerprint originali, ricalcola sullo stesso campione i punteggi delle predizioni storiche Microsoft con la funzione ufficiale, genera le tabelle e crea la relazione. Se qualunque controllo fallisce, **non forzare lo script e non modificare i file originali**: conservare l'errore per analizzarlo.

In [ ]:
import os
import subprocess
from pathlib import Path

SCRIPT = '# EVALUATION20 — Relazione per il relatore, 9 novembre 2026\n# Eseguire in Google Colab nel runtime gia\' predisposto.\n# Legge esclusivamente risultati congelati e la baseline ufficiale Microsoft.\n# Nessuna API LLM, nessuna modifica delle 900 sorgenti, nessun nuovo esperimento.\n# CREA nuovi artefatti con apertura esclusiva (xb); NON rieseguire dopo successo.\n\nfrom __future__ import annotations\n\nimport csv\nimport hashlib\nimport importlib.util\nimport io\nimport json\nimport math\nimport re\nimport subprocess\nimport sys\nfrom collections import Counter\nfrom decimal import Decimal\nfrom pathlib import Path\nfrom random import Random\nfrom statistics import mean\nfrom urllib.request import Request, urlopen\n\nREPO = Path(\'/content/Tesi_OpenRCA_Evaluation20\')\nDATA = Path(\'/content/drive/MyDrive/Tesi_OpenRCA\')\nSRC = REPO / \'src\'\nif str(SRC) not in sys.path:\n    sys.path.insert(0, str(SRC))\nsys.dont_write_bytecode = True\n\nfrom evaluation import evaluation20_campaign as campaign\nfrom evaluation import evaluation20_preregister as registry\nfrom output.v2_task_contract import enforce_task_output\n\nGIT_COMMIT = \'e4845e7fe9a399e10dd2db6f320dfeaeeddf3379\'\nPLAN_SHA = \'d390d7acd3db6931c48a64bcd35757ebadf7a7dbdb12b718c1154b45e8757342\'\nFREEZE_SHA = \'f53effaa81fada14bab31b50ba0983e3642ff818294ac43550587e6982e4ee5f\'\nINVENTORY_SHA = \'c12257a0ba87502090121c6393182f5f94a22ef08cef8fadbb853c4bb5addd18\'\nSCORING_SHA = \'d2f7f6495a36c89664cb6cd864bc0f59dd55b4aa18cee29032b1d0c81748d780\'\nSCORER_COMMIT = \'2a481ae63cb045c3eadbeca9eddaa10ec1151979\'\nSCORER_SHA = \'9a1c235139266c32b0e95c7e9938591b8fe014aad4cde95580f73f13a54d53f1\'\nSCORER_PATH = Path(\'/content/OpenRCA_official_scoring_2a481ae63c/main/evaluate.py\')\nBASELINE_BLOBS = {\n    \'cloudbed-1\': \'b81775fd5ea8e8fca86daf2e2a2fbf4cfc0b8510\',\n    \'cloudbed-2\': \'080db1b08e2c6d600663da1d5850f4498195ef2c\',\n}\nVARIANTS = (\'A0\', \'A1\', \'A1_8R\', \'A1_8R_Planning\', \'A1_PAVE\')\nMODELS = (\'sonnet\', \'haiku\', \'gemini\')\nCONTRASTS = tuple(zip(VARIANTS[:-1], VARIANTS[1:]))\n\nPLAN_PATH = registry.destination(DATA)\nFREEZE_PATH = DATA / \'data/protocols/evaluation20\' / (campaign.PROTOCOL + \'_execution_freeze_v1.json\')\nSCORING_PATH = DATA / \'data/results/scoring/evaluation20\' / (campaign.PROTOCOL + \'_official_raw_scoring_v1.json\')\nSCORING_SIDECAR = SCORING_PATH.with_suffix(\'.sha256.json\')\nMANIFEST_PATH = DATA / \'data/processed/benchmark_inputs_market.csv\'\nMARKET_PATH = DATA / \'Market.zip\'\nOUTPUT_DIR = DATA / \'data/reports/relatore_2026-11-09\'\nFILES = {\n    \'relazione\': OUTPUT_DIR / \'EVALUATION20_RELAZIONE_RELATORE_2026-11-09_v1.md\',\n    \'riepilogo\': OUTPUT_DIR / \'EVALUATION20_RIEPILOGO_15_CONFIGURAZIONI_v1.csv\',\n    \'ablazioni\': OUTPUT_DIR / \'EVALUATION20_ABLAZIONI_APPAIATE_v1.csv\',\n    \'baseline\': OUTPUT_DIR / \'EVALUATION20_BASELINE_MICROSOFT_20_CASI_v1.csv\',\n    \'dettaglio\': OUTPUT_DIR / \'EVALUATION20_DETTAGLIO_300_TENTATIVI_v1.csv\',\n    \'integrita\': OUTPUT_DIR / \'EVALUATION20_MANIFEST_REPORT_v1.json\',\n}\n\n\ndef sha_bytes(data: bytes) -> str:\n    return hashlib.sha256(data).hexdigest()\n\n\ndef sha_path(path: Path) -> str:\n    return sha_bytes(path.read_bytes())\n\n\ndef git_blob(data: bytes) -> str:\n    return hashlib.sha1(b\'blob \' + str(len(data)).encode(\'ascii\') + b\'\\0\' + data).hexdigest()\n\n\ndef assert_equal(actual, expected, what):\n    if actual != expected:\n        raise RuntimeError(f\'STOP {what}: ottenuto {actual!r}, previsto {expected!r}\')\n\n\ndef csv_data(items: list[dict], columns: list[str]) -> bytes:\n    buffer = io.StringIO(newline=\'\')\n    writer = csv.DictWriter(buffer, fieldnames=columns, extrasaction=\'ignore\', lineterminator=\'\\n\')\n    writer.writeheader()\n    writer.writerows(items)\n    return buffer.getvalue().encode(\'utf-8-sig\')\n\n\ndef clean(text):\n    return \' \'.join(str(text).split())\n\n\ndef boot(deltas, seed, b=10000):\n    rng = Random(seed)\n    vals = sorted(mean(rng.choices(deltas, k=len(deltas))) for _ in range(b))\n    return vals[int(0.025 * (b - 1))], vals[int(0.975 * (b - 1))]\n\n\ndef f4(value):\n    return f\'{value:.4f}\'\n\n\ndef signed(value):\n    return f\'{value:+.4f}\'\n\n\ndef rows_csv(buf):\n    return list(csv.DictReader(io.StringIO(buf.decode(\'utf-8-sig\'), newline=\'\')))\n\n\ndef load_microsoft_archive(cloudbed: str):\n    filename = f\'agent-Market-{cloudbed}.csv\'\n    url = (\n        \'https://raw.githubusercontent.com/microsoft/OpenRCA/\' +\n        SCORER_COMMIT + \'/rca/archive/\' + filename\n    )\n    content = None\n    try:\n        with urlopen(Request(url, headers={\'User-Agent\': \'EVALUATION20-thesis-audit\'}), timeout=45) as response:\n            content = response.read()\n        assert_equal(git_blob(content), BASELINE_BLOBS[cloudbed], \'blob archivio Microsoft\')\n    except Exception as exc:\n        content = None\n        # Sorgente alternativa solo se i byte coincidono con l\'archivio fissato.\n        for p in (DATA / filename, DATA / \'data\' / filename,\n                  Path(\'/content/OpenRCA_official\') / \'rca/archive\' / filename,\n                  Path(\'/content/OpenRCA_official_scoring_2a481ae63c\') / \'rca/archive\' / filename):\n            if p.is_file() and git_blob(p.read_bytes()) == BASELINE_BLOBS[cloudbed]:\n                content = p.read_bytes()\n                print(\'  Archivio Microsoft verificato da copia locale:\', p)\n                break\n        if content is None:\n            raise RuntimeError(f\'STOP: impossibile scaricare o trovare archivio verificato {cloudbed}: {exc}\') from exc\n    rows = rows_csv(content)\n    assert_equal(len(rows), 70 if cloudbed == \'cloudbed-1\' else 78, \'numero righe archivio\')\n    assert {\'row_id\', \'score\', \'prediction\', \'instruction\', \'task_index\'} <= set(rows[0]), \'Colonne archivio mancanti\'\n    indexed = {}\n    for row in rows:\n        rid = Decimal(row[\'row_id\'])\n        assert rid == rid.to_integral_value(), \'row_id non intero\'\n        ix = int(rid)\n        assert ix not in indexed, \'row_id duplicato\'\n        indexed[ix] = row\n    assert_equal(set(indexed), set(range(len(rows))), \'sequenza row_id archivio\')\n    return indexed, sha_bytes(content)\n\n\ndef main():\n    print(\'=== RELAZIONE RELATORE — EVALUATION20 ===\', flush=True)\n    if any(p.exists() for p in FILES.values()):\n        raise FileExistsError(\'STOP: report precedente presente. Nessun file sara\\u2019 sovrascritto.\')\n    assert_equal(registry.git_head_clean(REPO), GIT_COMMIT, \'commit Git\')\n    for p, h, title in ((PLAN_PATH, PLAN_SHA, \'piano\'), (FREEZE_PATH, FREEZE_SHA, \'freeze\'),\n                        (SCORING_PATH, SCORING_SHA, \'scoring\'), (SCORER_PATH, SCORER_SHA, \'scorer\')):\n        assert_equal(sha_path(p), h, title)\n    sidecar = json.loads(SCORING_SIDECAR.read_text(encoding=\'utf-8\'))\n    assert_equal(sidecar[\'scoring_report_sha256\'], SCORING_SHA, \'sidecar scoring\')\n    plan = json.loads(PLAN_PATH.read_text(encoding=\'utf-8\'))\n    freeze = json.loads(FREEZE_PATH.read_text(encoding=\'utf-8\'))\n    report = json.loads(SCORING_PATH.read_text(encoding=\'utf-8\'))\n    state = campaign.preflight(DATA, REPO)\n    assert_equal(state[\'counts\'], {\'MODEL_ERROR_REVIEW_NO_RETRY\': 5, \'complete\': 295}, \'stati\')\n    assert len(state[\'runs\']) == 300 and all(not r[\'issues\'] for r in state[\'runs\'])\n    assert_equal(freeze[\'inventory_sha256\'], INVENTORY_SHA, \'inventario\')\n    assert_equal(report[\'planned_attempts\'], 300, \'tentativi\')\n    assert_equal(report[\'scored_attempts\'], 295, \'score disponibili\')\n    assert_equal(report[\'provider_errors_not_scored\'], 5, \'errori provider\')\n    assert_equal(report[\'prediction_field\'], \'raw_response\', \'scoring raw\')\n    assert_equal(report[\'official_scorer_source_sha256\'], SCORER_SHA, \'sorgente scorer report\')\n    assert_equal(sha_path(MANIFEST_PATH), plan[\'manifest_sha256\'], \'manifest GT-free\')\n    assert_equal(len(plan[\'cases\']), 20, \'casi piano\')\n    assert_equal(len(plan[\'runs\']), 300, \'run piano\')\n\n    # Controllo indipendente dei 900 artefatti e impronta aggregata.\n    inventory = hashlib.sha256()\n    for item in sorted(state[\'runs\'], key=lambda r: r[\'run_id\']):\n        for kind, path in zip((\'claim\',\'result\',\'checksum\'), campaign.paths(DATA, item[\'run_id\'])):\n            dg = sha_path(path)\n            inventory.update(f"{item[\'run_id\']}\\t{kind}\\t{dg}\\n".encode(\'utf-8\'))\n    assert_equal(inventory.hexdigest(), INVENTORY_SHA, \'fingerprint dei 900 file\')\n    print(\'[1/6] Freeze, scorer, scoring, 900 file verificati: OK\', flush=True)\n\n    spec = importlib.util.spec_from_file_location(\'microsoft_openrca_relazione\', SCORER_PATH)\n    assert spec and spec.loader\n    scorer = importlib.util.module_from_spec(spec)\n    spec.loader.exec_module(scorer)\n\n    with MANIFEST_PATH.open(\'r\', encoding=\'utf-8-sig\', newline=\'\') as stream:\n        gtfree = {r[\'case_id\']: r for r in csv.DictReader(stream)}\n    case_meta = {r[\'case_id\']: r for r in plan[\'cases\']}\n    assert_equal(len(case_meta), 20, \'casi distinti\')\n    import zipfile\n    query = {}\n    with zipfile.ZipFile(MARKET_PATH, \'r\') as package:\n        for cb in (\'cloudbed-1\', \'cloudbed-2\'):\n            raw = package.read(f\'Market/{cb}/query.csv\')\n            query[cb] = rows_csv(raw)\n            assert {\'instruction\', \'task_index\', \'scoring_points\'} <= set(query[cb][0])\n\n    archived = {}\n    archive_sha = {}\n    for cb in (\'cloudbed-1\', \'cloudbed-2\'):\n        archived[cb], archive_sha[cb] = load_microsoft_archive(cb)\n\n    base_cases = {}\n    baseline_table = []\n    for case in plan[\'cases\']:\n        cid, cb = case[\'case_id\'], case[\'cloudbed\']\n        meta = gtfree[cid]\n        qi = int(meta[\'query_index\'])\n        original = query[cb][qi]\n        old = archived[cb][qi]\n        for name, value in ((\'task manifest\', original[\'task_index\']), (\'task archive\', old[\'task_index\'])):\n            assert_equal(value.strip(), case[\'task_index\'], f\'{cid} {name}\')\n        assert_equal(clean(meta[\'instruction\']), clean(original[\'instruction\']), f\'{cid} istruzione manifest\')\n        assert_equal(clean(old[\'instruction\']), clean(original[\'instruction\']), f\'{cid} istruzione archivio Microsoft\')\n        assert original[\'scoring_points\'].strip(), \'ground truth vuoto\'\n        passed, failed, recalculated = scorer.evaluate(old[\'prediction\'], original[\'scoring_points\'])\n        published = float(old[\'score\'])\n        if not math.isfinite(published) or abs(recalculated - published) > 0.00501:\n            raise RuntimeError(f\'STOP: score archivio e funzione ufficiale non coincidono {cid}: {published} vs {recalculated}\')\n        base_cases[cid] = float(recalculated)\n        baseline_table.append({\n            \'case_id\': cid, \'cloudbed\': cb, \'query_index\': qi, \'task_index\': case[\'task_index\'],\n            \'rca_agent_archived_score\': published, \'rca_agent_recomputed_score\': float(recalculated),\n            \'rca_agent_strict\': int(recalculated == 1.0),\n            \'archive_git_blob\': BASELINE_BLOBS[cb],\n        })\n    assert_equal(len(base_cases), 20, \'20 score baseline\')\n    baseline_mean = mean(base_cases.values())\n    baseline_strict = sum(v == 1 for v in base_cases.values())\n    print(\'[2/6] Baseline RCA-agent Microsoft: allineamento 20/20, scorer ricalcolato OK\')\n    print(\'      Media Microsoft:\', f4(baseline_mean), \'| Strict:\', f\'{baseline_strict}/20\', flush=True)\n\n    status_by_id = {r[\'run_id\']: r for r in state[\'runs\']}\n    indexed = {}\n    all_rows = []\n    invalid_count, wrapper_only, positive_invalid, strict_invalid = 0, 0, 0, 0\n    FENCE = re.compile(r\'\\s*```(?:json)?[ \\t]*\\r?\\n(?P<body>.*?)\\s*```\\s*\', re.I | re.S)\n    for row in report[\'runs\']:\n        rid, cid = row[\'run_id\'], row[\'case_id\']\n        assert rid in status_by_id and cid in case_meta\n        meta = status_by_id[rid]\n        for k in (\'case_id\', \'variant\', \'model_key\', \'status\'):\n            assert_equal(row[k], meta[k], f\'{rid} {k}\')\n        _, source_path, _ = campaign.paths(DATA, rid)\n        assert_equal(sha_path(source_path), row[\'source_result_sha256\'], f\'{rid} result hash\')\n        src = json.loads(source_path.read_text(encoding=\'utf-8\'))\n        key = (row[\'model_key\'], cid, row[\'variant\'])\n        assert key not in indexed\n        complete = row[\'status\'] == \'complete\'\n        rec = {\n            \'run_id\': rid, \'case_id\': cid, \'cloudbed\': case_meta[cid][\'cloudbed\'],\n            \'task_index\': case_meta[cid][\'task_index\'], \'model\': row[\'model_key\'],\n            \'harness\': row[\'variant\'], \'status\': row[\'status\'],\n            \'official_raw_score\': None, \'strict\': None, \'contract_v2_valid\': None,\n            \'json_status\': src.get(\'json_status\'), \'total_tokens\': None,\n            \'end_to_end_seconds\': None, \'rca_agent_same_case_score\': base_cases[cid],\n            \'operational_strict\': None, \'result_sha256\': row[\'source_result_sha256\'],\n        }\n        if complete:\n            score = row[\'official_raw_score\']\n            assert type(score) in (float, int) and math.isfinite(score) and 0 <= score <= 1\n            audit = src[\'output_contract_v2\']\n            assert type(audit[\'valid\']) is bool\n            assert src[\'metrics\'][\'token_usage_complete\'] is True\n            assert row[\'raw_response_sha256\'] == sha_bytes(src[\'raw_response\'].encode(\'utf-8\'))\n            m = src[\'metrics\']\n            rec.update({\n                \'official_raw_score\': float(score), \'strict\': int(score == 1.0),\n                \'contract_v2_valid\': int(audit[\'valid\']), \'total_tokens\': int(m[\'total_tokens\']),\n                \'end_to_end_seconds\': float(m[\'end_to_end_seconds\']),\n                \'operational_strict\': int(score == 1.0 and audit[\'valid\']),\n            })\n            if not audit[\'valid\']:\n                invalid_count += 1\n                positive_invalid += int(score > 0)\n                strict_invalid += int(score == 1.0)\n                mat = FENCE.fullmatch(src[\'raw_response\'])\n                if mat:\n                    case = case_meta[cid]\n                    _, status, inner_audit = enforce_task_output(\n                        mat.group(\'body\'), task_index=case[\'task_index\'],\n                        instruction=gtfree[cid][\'instruction\'],\n                        start_unix=int(case[\'start_unix\']), end_unix=int(case[\'end_unix\'])\n                    )\n                    wrapper_only += int(status == \'valid_json\' and inner_audit[\'valid\'] is True)\n        else:\n            assert_equal(row[\'status\'], \'MODEL_ERROR_REVIEW_NO_RETRY\', f\'{rid} stato errore\')\n            assert row[\'official_raw_score\'] is None and src.get(\'error\') is not None\n        all_rows.append(rec)\n        indexed[key] = rec\n    assert_equal(len(indexed), 300, \'300 chiavi uniche\')\n    assert_equal(sum(x[\'official_raw_score\'] is not None for x in all_rows), 295, \'295 scoring\')\n    assert_equal(invalid_count, 80, \'80 non conformi\')\n    assert_equal(wrapper_only, 68, \'68 soli wrapper\')\n    assert_equal(positive_invalid, 27, \'27 non conformi con score positivo\')\n    assert_equal(strict_invalid, 13, \'13 non conformi strict\')\n    print(\'[3/6] Audit score, 503, conformita\\u2019 e wrapper: OK\', flush=True)\n\n    # Riepilogo per 15 condizioni, con baseline sugli esatti incidenti disponibili.\n    table = []\n    for mod in MODELS:\n        for variant in VARIANTS:\n            group = [r for r in all_rows if r[\'model\'] == mod and r[\'harness\'] == variant]\n            good = [r for r in group if r[\'official_raw_score\'] is not None]\n            assert_equal(len(group), 20, \'20 tentativi per configurazione\')\n            base_matched = mean(x[\'rca_agent_same_case_score\'] for x in good)\n            mean_own = mean(x[\'official_raw_score\'] for x in good)\n            diffs = [x[\'official_raw_score\'] - x[\'rca_agent_same_case_score\'] for x in good]\n            table.append({\n                \'model\': mod, \'harness\': variant, \'attempts\': len(group), \'complete\': len(good),\n                \'provider_errors\': len(group)-len(good), \'official_mean\': mean_own,\n                \'strict\': sum(x[\'strict\'] for x in good),\n                \'v2_valid\': sum(x[\'contract_v2_valid\'] for x in good),\n                \'operational_strict\': sum(x[\'operational_strict\'] for x in good),\n                \'mean_tokens\': mean(x[\'total_tokens\'] for x in good),\n                \'mean_seconds\': mean(x[\'end_to_end_seconds\'] for x in good),\n                \'rca_agent_matched_mean\': base_matched,\n                \'delta_vs_rca_agent\': mean(diffs),\n                \'wins_vs_rca_agent\': sum(d > 1e-12 for d in diffs),\n                \'ties_vs_rca_agent\': sum(abs(d) <= 1e-12 for d in diffs),\n                \'losses_vs_rca_agent\': sum(d < -1e-12 for d in diffs),\n            })\n\n    pairs = []\n    for mi, mod in enumerate(MODELS):\n        for ci, (old, new) in enumerate(CONTRASTS):\n            chosen = [(indexed[(mod, cid, old)], indexed[(mod, cid, new)])\n                      for cid in sorted(base_cases)\n                      if indexed[(mod,cid,old)][\'official_raw_score\'] is not None\n                      and indexed[(mod,cid,new)][\'official_raw_score\'] is not None]\n            diffs = [b[\'official_raw_score\']-a[\'official_raw_score\'] for a,b in chosen]\n            lo,hi = boot(diffs, seed=20261108 + 100*mi + ci)\n            avg_old_tok = mean(a[\'total_tokens\'] for a,b in chosen)\n            avg_new_tok = mean(b[\'total_tokens\'] for a,b in chosen)\n            avg_old_sec = mean(a[\'end_to_end_seconds\'] for a,b in chosen)\n            avg_new_sec = mean(b[\'end_to_end_seconds\'] for a,b in chosen)\n            pairs.append({\n                \'model\': mod, \'old\': old, \'new\': new, \'matched_n\': len(diffs),\n                \'delta_mean\': mean(diffs), \'ci95_low\':lo, \'ci95_high\':hi,\n                \'wins\':sum(d>1e-12 for d in diffs),\n                \'ties\':sum(abs(d)<=1e-12 for d in diffs),\n                \'losses\':sum(d<-1e-12 for d in diffs),\n                \'token_change_pct\':100*(avg_new_tok/avg_old_tok-1),\n                \'time_change_pct\':100*(avg_new_sec/avg_old_sec-1),\n            })\n\n    common_gemini = [cid for cid in sorted(base_cases)\n                     if all(indexed[(\'gemini\',cid,v)][\'official_raw_score\'] is not None for v in VARIANTS)]\n    assert_equal(len(common_gemini),16,\'Gemini comune\')\n    print(\'[4/6] 15 configurazioni, 12 confronti e sottocoorte Gemini: OK\', flush=True)\n\n    # Relazione per il professore; il report non espone scoring_points/raw_response.\n    md = [\n        \'# Relazione scientifica EVALUATION20 — incontro con il relatore\',\n        \'\', \'**Data:** 9 novembre 2026  |  **Scadenza tesi:** 1 dicembre 2026\',\n        \'\', \'## Domanda di ricerca e progetto\',\n        \'\', \'La tesi studia l\\\'effetto di livelli progressivi di Harness Engineering sulla Root Cause Analysis con LLM,\',\n        \'utilizzando il benchmark Microsoft OpenRCA 1.0. Le configurazioni sono A0 (tool/LLM), A1 (evidenze deterministiche),\',\n        \'A1_8R (otto regole), A1_8R_Planning (esplorazione e verifica) e A1_PAVE (evidenze strutturali-temporali).\',\n        \'A1_PAVE e\\u2019 un adattamento sperimentale ispirato a PAVE, non un\\u2019implementazione integrale di OpenRCA 2.0.\',\n        \'\', \'## Disegno, protezioni e stato\',\n        \'\', \'- Development: 8 incidenti, separati dai dati di valutazione; EVALUATION20 non e\\u2019 una replica dei development.\',\n        \'- Evaluation: 20 incidenti (10 per cloudbed), sette task, casi multi-guasto, disgiunzione temporale con coorti protette.\',\n        \'- 20 incidenti x 3 modelli x 5 harness x 1 replica = **300 tentativi**.\',\n        \'- **295 diagnosi complete**; **5 errori provider Gemini 503**, conservati senza retry, con score mancante e non imputato.\',\n        \'- Protocollo preregistrato, commit e 900 artefatti verificati mediante SHA256; scoring eseguito dopo il freeze.\',\n        \'- Metriche primarie: score sul `raw_response` con `evaluate()` originale Microsoft. Il contratto V2 e\\u2019 un controllo separato del formato.\',\n        \'\', \'## Risultati ufficiali e confronto RETROSPETTIVO Microsoft\',\n        \'\', f\'Baseline Microsoft RCA-agent archiviata, sui **medesimi 20 incidenti**: **score medio {baseline_mean:.4f}**, **strict {baseline_strict}/20**.\',\n        \'I punteggi delle predizioni storiche sono stati ricalcolati con lo scorer ufficiale e allineati ai `query_index` originali.\',\n        \'\', \'| Modello | Harness | Complete /20 | Score nostro | Strict | V2 valide | Strict e V2 | Baseline su casi disponibili | Delta vs Microsoft |\',\n        \'|---|---|---:|---:|---:|---:|---:|---:|---:|\',\n    ]\n    for r in table:\n        md.append(f"| {r[\'model\']} | {r[\'harness\']} | {r[\'complete\']}/20 | {f4(r[\'official_mean\'])} | {r[\'strict\']} | {r[\'v2_valid\']} | {r[\'operational_strict\']} | {f4(r[\'rca_agent_matched_mean\'])} | {signed(r[\'delta_vs_rca_agent\'])} |")\n    md += [\n        \'\', \'**ATTENZIONE — Interpretazione della baseline:** il confronto con RCA-agent Microsoft usa gli stessi casi e la stessa\',\n        \'funzione di scoring, ma non lo stesso modello, set di tool, condizioni temporali o budget di esecuzione.\',\n        \'Non e\\u2019 un confronto controllato che isoli l\\u2019effetto dell\\u2019harness. Il risultato non e\\u2019 generalizzabile a tutti gli incidenti OpenRCA.\',\n        \'I punteggi Microsoft su 20 casi non vanno confrontati con le percentuali pubblicate sull\\u2019intero benchmark.\',\n        \'\', \'## Studio di ablazione: differenze appaiate per incidente\',\n        \'\', \'Differenza positiva: il nuovo harness ha score maggiore. IC bootstrap percentile al 95%, 10.000 ricampionamenti di incidenti, esplorativi e non corretti per confronti multipli.\',\n        \'\', \'| Modello | Ablazione | Coppie | Delta score | IC95% | W/T/L | Delta token % | Delta tempo % |\',\n        \'|---|---|---:|---:|---|---|---:|---:|\',\n    ]\n    for r in pairs:\n        md.append(f"| {r[\'model\']} | {r[\'old\']} -> {r[\'new\']} | {r[\'matched_n\']} | {signed(r[\'delta_mean\'])} | [{signed(r[\'ci95_low\'])}; {signed(r[\'ci95_high\'])}] | {r[\'wins\']}/{r[\'ties\']}/{r[\'losses\']} | {r[\'token_change_pct\']:+.1f}% | {r[\'time_change_pct\']:+.1f}% |")\n    md += [\n        \'\', \'Il **confronto primario preregistrato** e\\u2019 A1 - A0 entro ciascun modello. Le altre tre ablazioni sono secondarie.\',\n        \'Il dato principale e\\u2019 un guadagno positivo per Sonnet; gli intervalli per Haiku e Gemini includono lo zero.\',\n        \'Regole e planning non hanno effetti universalmente positivi; le combinazioni migliori cambiano per modello.\',\n        \'\', \'## Affidabilita\\u2019 del formato e disponibilita\\u2019\',\n        \'\', f\'- Contratto V2: **{295-invalid_count}/295 valide**, **{invalid_count}/295 non conformi**.\',\n        f\'- Tra le non conformi, **{wrapper_only}/{invalid_count}** supererebbero il contratto rimuovendo *solo* un wrapper Markdown completo.\',\n        f\'- **{positive_invalid}** non conformi ricevono score Microsoft > 0; **{strict_invalid}** hanno score 1.0 ma non rispettano V2.\',\n        \'- Queste sono diagnosi NON corrette retroattivamente: lo score ufficiale e\\u2019 rimasto quello della risposta originale.\',\n        \'- La metrica esplorativa `strict e V2` indica successo diagnostico e conformita\\u2019 simultanei, NON sostituisce lo score Microsoft.\',\n        \'- Gli errori provider 503 sono mancata disponibilita\\u2019, non diagnosi errate; nessuna imputazione automatica a zero.\',\n        \'\', \'## Limiti metodologici\',\n        \'\', \'1. **N = 20 incidenti**, con una sola replica per modello/harness/incidente: potenza limitata e varianza dell\\u2019LLM non misurata.\',\n        \'2. Le 300 run non sono 300 incidenti indipendenti; usare confronti appaiati per caso.\',\n        \'3. Coorte selezionata con criteri GT-free, ma non e\\u2019 un campione casuale rappresentativo di tutti gli incidenti.\',\n        \'4. Confronti multipli, intervalli bootstrap esplorativi, molti pareggi; non dichiarare significativita\\u2019 confermativa universale.\',\n        \'5. I cinque errori Gemini comportano missing data: non assumere missing completely at random.\',\n        \'6. Confronto con RCA-agent storico non controllato per modello, strumenti, epoca e budget.\',\n        \'7. Percezione di efficienza: token comparabili soprattutto *entro* provider; tempi dipendono anche da infrastruttura e latenza.\',\n        \'8. Il contratto V2 misura struttura e cardinalita\\u2019, non correttezza causale; PAVE non dimostra causalita\\u2019 certificata.\',\n        \'\', \'## Domande per il professore (revisione del 9 novembre)\',\n        \'\', \'1. E\\u2019 corretta la formulazione della conclusione primaria: evidenze deterministiche utili su Sonnet, effetti dipendenti dal modello?\',\n        \'2. Presentiamo il confronto con RCA-agent storico come **benchmark contestuale**, non come confronto causale?\',\n        \'3. Le osservazioni OOD (4 classi di guasto non codificate nelle 8 regole) vanno esplorate prima della consegna o come futuro sviluppo?\',\n        \'4. Il possibile knowledge transfer con approvazione umana resta future work, anziche\\u2019 essere sviluppato prima della consegna?\',\n        \'\', \'## Provenienza e riferimenti\',\n        \'\', f\'- Repository della tesi: https://github.com/alessio1996A/Tesi_Laurea_Magistrale_OpenRCA (commit `{GIT_COMMIT}`)\',\n        f\'- Scorer e archivio Microsoft: https://github.com/microsoft/OpenRCA/tree/{SCORER_COMMIT}\',\n        f\'- Piano SHA256: `{PLAN_SHA}`\',\n        f\'- Freeze SHA256: `{FREEZE_SHA}`\',\n        f\'- Inventario dei 900 artefatti SHA256: `{INVENTORY_SHA}`\',\n        f\'- Scoring nostro SHA256: `{SCORING_SHA}`\',\n        f\'- Scorer Microsoft SHA256: `{SCORER_SHA}`\',\n        f"- Archive cloudbed-1: SHA256 `{archive_sha[\'cloudbed-1\']}`; Git blob `{BASELINE_BLOBS[\'cloudbed-1\']}`",\n        f"- Archive cloudbed-2: SHA256 `{archive_sha[\'cloudbed-2\']}`; Git blob `{BASELINE_BLOBS[\'cloudbed-2\']}`",\n        \'\', \'**Dichiarazione di integrita\\u2019:** nessun risultato di inferenza, scoring o freeze e\\u2019 stato modificato per produrre questa relazione.\',\n        \'\',\n    ]\n    text_bytes = (\'\\n\'.join(md)).encode(\'utf-8\')\n    payloads = {\n        \'relazione\': text_bytes,\n        \'riepilogo\': csv_data(table, list(table[0])),\n        \'ablazioni\': csv_data(pairs, list(pairs[0])),\n        \'baseline\': csv_data(baseline_table, list(baseline_table[0])),\n        \'dettaglio\': csv_data(all_rows, list(all_rows[0])),\n    }\n    inventory_manifest = {\n        \'schema\': \'evaluation20_professor_report_provenance_v1\',\n        \'source\': \'frozen evaluation20 results + exact Microsoft archive\',\n        \'git_commit\': GIT_COMMIT, \'plan_sha256\': PLAN_SHA,\n        \'execution_freeze_sha256\': FREEZE_SHA, \'inventory_sha256\': INVENTORY_SHA,\n        \'scoring_report_sha256\': SCORING_SHA, \'scorer_commit\': SCORER_COMMIT,\n        \'scorer_sha256\': SCORER_SHA,\n        \'microsoft_archive_blob_sha1\': BASELINE_BLOBS,\n        \'microsoft_archive_sha256\': archive_sha,\n        \'baseline_mean_recomputed\': baseline_mean,\n        \'baseline_strict_on_20\': baseline_strict,\n        \'outputs\': {FILES[k].name: {\'sha256\': sha_bytes(v), \'bytes\': len(v)} for k,v in payloads.items()},\n        \'provider_api_calls\': 0, \'previous_results_modified\': False,\n    }\n    payloads[\'integrita\'] = (\n        json.dumps(inventory_manifest, indent=2, sort_keys=True, ensure_ascii=False, allow_nan=False)+\'\\n\'\n    ).encode(\'utf-8\')\n    assert all(not path.exists() for path in FILES.values())\n    print(\'[5/6] Relazione e tabelle predisposte (nessuna modifica alle sorgenti): OK\', flush=True)\n    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)\n    for k in (\'relazione\',\'riepilogo\',\'ablazioni\',\'baseline\',\'dettaglio\',\'integrita\'):\n        with FILES[k].open(\'xb\') as stream:\n            stream.write(payloads[k])\n        assert_equal(sha_path(FILES[k]), sha_bytes(payloads[k]), f\'{k} readback\')\n    assert_equal(sha_path(FREEZE_PATH), FREEZE_SHA, \'freeze dopo scrittura\')\n    assert_equal(sha_path(SCORING_PATH), SCORING_SHA, \'scoring dopo scrittura\')\n    print(\'[6/6] Sei artefatti create-only, readback SHA256: OK\', flush=True)\n    print(\'\\n=== MATERIALI PER IL PROFESSORE ===\')\n    for k, p in FILES.items():\n        print(f\'{k}: {p}\')\n    print(\'\\nBaseline RCA-agent sullo stesso insieme (score ricalcolati):\', f4(baseline_mean),\n          \'| strict\', baseline_strict, \'/20\')\n    print(\'\\nModello     Harness               Nostro   Microsoft matched   Delta  strict\')\n    for r in table:\n        print(f"{r[\'model\']:<11} {r[\'harness\']:<21} {r[\'official_mean\']:.4f}   "\n              f"{r[\'rca_agent_matched_mean\']:.4f}             {r[\'delta_vs_rca_agent\']:+.4f}  {r[\'strict\']}")\n    print(\'\\n300 tentativi, 295 complete, 5 provider errors | 80 non conformi V2 | 68 wrapper-only\')\n    print(\'Nessun retry, nessuna API LLM, nessuna modifica alle 900 sorgenti.\')\n    print(\'CODICE USCITA: 0\')\n\n\nif __name__ == \'__main__\':\n    main()\n'
ENV = os.environ.copy()
ENV['PYTHONPATH'] = str(Path('/content/Tesi_OpenRCA_Evaluation20/src'))
ENV['PYTHONDONTWRITEBYTECODE'] = '1'
PROCESS = subprocess.run(
    ['/content/evaluation20_gemini_venv/bin/python', '-B', '-u', '-c', SCRIPT],
    cwd='/content/Tesi_OpenRCA_Evaluation20', env=ENV,
    capture_output=True, text=True
)
print(PROCESS.stdout)
if PROCESS.returncode:
    print(PROCESS.stderr)
    raise RuntimeError('STOP: report non confermato; non rieseguire automaticamente')
print('Codice uscita:', PROCESS.returncode)

## Lettura del report per il professore

Apri su Google Drive la cartella `Tesi_OpenRCA/data/reports/relatore_2026-11-09/` e consulta il file `EVALUATION20_RELAZIONE_RELATORE_2026-11-09_v1.md`.

**Tre messaggi da discutere:** (1) evidenze deterministiche A1 migliorano la baseline interna A0, soprattutto su Sonnet; (2) la progressione dell'harness non è monotona e mostra trade-off fra accuratezza/risorse; (3) la baseline Microsoft RCA-agent storica, ricalcolata sugli stessi incidenti, è un confronto descrittivo e non controllato, a causa delle differenze di modelli e condizioni sperimentali.

Il contratto V2 misura il **formato**, mentre il punteggio primario è quello dello **scorer ufficiale Microsoft**. I cinque errori provider non sono diagnosi errate e non vengono imputati automaticamente a zero.